# 03 - Matplotlib 可视化基础：把数据和模型“画出来”

> 配套脚本：`02_python_ai_tools/03_matplotlib_basics.py`

本 Notebook 不是对原脚本的逐行搬运，而是围绕“AI 开发中为什么需要可视化、如何选择图表、如何写出可复用绘图代码”重新组织的教程版。

## 本 Notebook 概览

| 章节 | 内容 | 你将理解 |
|------|------|----------|
| 1 | 为什么 AI 需要可视化 | 可视化在数据探索、训练监控、评估和调试中的作用 |
| 2 | Matplotlib 基本模型 | Figure、Axes、plot、style、中文字体、网格、图例 |
| 3 | 常用图表类型 | 折线图、散点图、直方图、柱状图、箱线图、子图布局 |
| 4 | 数据分布与关系 | 如何看分布、相关性、类别边界、异常点 |
| 5 | 训练过程可视化 | loss/accuracy 曲线、过拟合、早停点、学习率影响 |
| 6 | 模型评估图 | 混淆矩阵、ROC 曲线、PR 曲线、阈值选择 |
| 7 | 神经网络常见图 | 激活函数、梯度下降路径、决策边界、注意力热力图 |
| 8 | 绘图工程习惯 | 封装函数、保存图片、避免常见错误 |
| 9 | 总结与练习 | 检查清单与扩展练习 |

**先决条件**：安装 `numpy`、`matplotlib`。部分章节如果检测到 `scikit-learn` 会使用它，否则自动使用 NumPy 版本的替代实现。

```bash
pip install numpy matplotlib scikit-learn
```

**核心主线**：

可视化不是“美化结果”，而是 AI 开发中的调试工具：先看数据，再看训练，再看评估，最后看模型内部是否符合预期。


In [ ]:
# 环境准备
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# 让图在 Notebook 中直接显示
%matplotlib inline

# 中文字体设置：不同系统可用字体不同，Matplotlib 会自动选择可用项
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

print('NumPy 版本:', np.__version__)
print('Matplotlib 版本:', plt.matplotlib.__version__)
print('✅ 环境准备完毕')


## 1. 为什么 AI 开发离不开可视化？

在 AI 项目中，可视化通常服务于四类问题：

| 阶段 | 典型问题 | 常用图表 |
|------|----------|----------|
| 数据探索 | 特征分布是否异常？类别是否不均衡？变量之间是否相关？ | 直方图、箱线图、散点图、热力图 |
| 训练监控 | loss 是否下降？是否过拟合？学习率是否合适？ | 折线图、双轴图、平滑曲线 |
| 模型评估 | 哪些类别容易混淆？不同阈值下表现如何？ | 混淆矩阵、ROC、PR 曲线 |
| 模型调试 | 激活函数是否饱和？梯度是否震荡？注意力集中在哪里？ | 函数图、等高线、热力图 |

一个很实用的经验是：

> 如果一个模型表现不好，先不要急着调参；先把数据、训练过程和错误样本画出来。


## 2. Matplotlib 的基本模型：Figure 与 Axes

Matplotlib 有两个核心对象：

- **Figure**：整张画布，可以理解为“一页纸”
- **Axes**：画布中的一个坐标系，可以理解为“一张图”

推荐使用面向对象写法：

```python
fig, ax = plt.subplots()
ax.plot(x, y)
ax.set_title('标题')
```

相比直接使用 `plt.plot()`，这种写法在多子图、复杂布局、封装函数时更清晰。


In [ ]:
# 一个最小但完整的 Matplotlib 示例：sin 与 cos
x = np.linspace(0, 2*np.pi, 300)
y_sin = np.sin(x)
y_cos = np.cos(x)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(x, y_sin, label='sin(x)', color='C0', linewidth=2)
ax.plot(x, y_cos, label='cos(x)', color='C3', linewidth=2, linestyle='--')
ax.set_xlabel('x')
ax.set_ylabel('函数值')
ax.set_title('三角函数：折线图的基本组成')
ax.legend()
ax.grid(alpha=0.3)
plt.show()


### 2.1 一张图通常由哪些元素组成？

一张容易理解的图通常至少包含：

1. 标题：这张图想说明什么？
2. 坐标轴标签：横轴和纵轴分别是什么？单位是什么？
3. 图例：多条线或多组点分别代表什么？
4. 网格：帮助读数，但透明度不要太高
5. 合理的颜色和线型：颜色负责区分，线型可以强化含义

下面把这些元素拆开观察。


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)

# 1. 缺少上下文：不推荐
axes[0].plot(x, y_sin)
axes[0].plot(x, y_cos)
axes[0].set_title('信息不足')

# 2. 加上标题和坐标轴
axes[1].plot(x, y_sin)
axes[1].plot(x, y_cos)
axes[1].set_title('有标题和坐标轴')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].grid(alpha=0.3)

# 3. 完整图表
axes[2].plot(x, y_sin, label='sin(x)', linewidth=2)
axes[2].plot(x, y_cos, label='cos(x)', linestyle='--', linewidth=2)
axes[2].set_title('完整可读版本')
axes[2].set_xlabel('x')
axes[2].grid(alpha=0.3)
axes[2].legend()

plt.suptitle('同样的数据，不同的图表表达质量', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. 常用图表类型：先选对图，再谈美观

不同图表适合回答不同问题：

| 图表 | 适合回答的问题 | AI 中的典型场景 |
|------|----------------|----------------|
| 折线图 | 一个量如何随时间/步骤变化？ | loss 曲线、accuracy 曲线、学习率变化 |
| 散点图 | 两个变量之间是否有关联？ | 特征与标签关系、聚类结果 |
| 直方图 | 一个变量的分布形状如何？ | 特征分布、残差分布、置信度分布 |
| 柱状图 | 不同类别的数量或指标如何比较？ | 类别样本数、模型指标对比 |
| 箱线图 | 分布的中位数、离散程度和异常点如何？ | 特征异常值检查、不同类别分布对比 |
| 热力图 | 矩阵或二维强度如何分布？ | 混淆矩阵、相关系数、注意力权重 |


In [ ]:
# 构造一份模拟数据，后面多个图表会复用
rng = np.random.default_rng(42)
heights = rng.normal(loc=170, scale=9, size=800)
weights = 0.55 * heights - 35 + rng.normal(0, 6, size=800)
classes = rng.choice(['猫', '狗', '兔子', '鸟'], size=800, p=[0.42, 0.33, 0.18, 0.07])

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# 直方图：看单变量分布
axes[0, 0].hist(heights, bins=30, color='skyblue', edgecolor='black', alpha=0.75)
axes[0, 0].axvline(heights.mean(), color='red', linestyle='--', label=f'均值={heights.mean():.1f}')
axes[0, 0].set_title('直方图：身高分布')
axes[0, 0].set_xlabel('身高（cm）')
axes[0, 0].legend()

# 散点图：看两个变量关系
axes[0, 1].scatter(heights, weights, alpha=0.45, s=18, color='coral')
axes[0, 1].set_title('散点图：身高与体重关系')
axes[0, 1].set_xlabel('身高（cm）')
axes[0, 1].set_ylabel('体重（kg）')

# 柱状图：看类别数量
labels, counts = np.unique(classes, return_counts=True)
axes[1, 0].bar(labels, counts, color=['C0', 'C1', 'C2', 'C3'])
axes[1, 0].set_title('柱状图：类别样本数量')
axes[1, 0].set_ylabel('样本数')
for i, c in enumerate(counts):
    axes[1, 0].text(i, c + 5, str(c), ha='center')

# 箱线图：看不同类别的体重分布
box_data = [weights[classes == label] for label in labels]
axes[1, 1].boxplot(box_data, tick_labels=labels, patch_artist=True)
axes[1, 1].set_title('箱线图：不同类别的体重分布')
axes[1, 1].set_ylabel('体重（kg）')

for ax in axes.flat:
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


### 3.1 子图布局：把相关信息放在一起比较

在 AI 教程或实验报告里，经常需要同时展示多张图：例如左边放数据，右边放模型预测；上面放 loss，下面放 accuracy。

常见布局函数：

- `plt.subplots(rows, cols)`：最常用
- `sharex=True` / `sharey=True`：共享坐标轴，适合对比
- `plt.tight_layout()`：自动调整间距
- `fig.suptitle()`：整张画布的大标题


In [ ]:
# 对比不同噪声强度下的线性关系
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharex=True, sharey=True)
x_feature = rng.normal(size=120)
noise_levels = [0.1, 0.5, 1.0, 2.0]

for ax, noise in zip(axes, noise_levels):
    y_target = 2 * x_feature + 1 + rng.normal(0, noise, size=len(x_feature))
    corr = np.corrcoef(x_feature, y_target)[0, 1]
    ax.scatter(x_feature, y_target, alpha=0.65)
    ax.set_title(f'噪声={noise}\n相关系数={corr:.2f}')
    ax.grid(alpha=0.3)
    ax.set_xlabel('Feature X')

axes[0].set_ylabel('Target y')
plt.suptitle('噪声越大，线性关系越不明显', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. 数据分布与关系：训练前先看数据

训练模型前至少应该检查：

1. 特征是否偏态严重？是否需要标准化或变换？
2. 标签是否类别不均衡？
3. 特征之间是否高度相关？
4. 是否存在明显异常点？
5. 不同类别是否有可分结构？

下面用一个二维分类数据集观察这些问题。


In [ ]:
# 生成二维分类数据：优先使用 sklearn；如果没有，则用 NumPy 手写一个替代版本
try:
    from sklearn.datasets import make_blobs
    X_blob, y_blob = make_blobs(n_samples=360, centers=3, cluster_std=[1.0, 1.4, 0.8], random_state=42)
    source = 'sklearn.datasets.make_blobs'
except Exception:
    centers = np.array([[-3, -1], [0.5, 2.5], [3, -1.5]])
    X_list, y_list = [], []
    for i, c in enumerate(centers):
        pts = rng.normal(loc=c, scale=[1.0 + 0.2*i, 0.8 + 0.3*i], size=(120, 2))
        X_list.append(pts)
        y_list.append(np.full(120, i))
    X_blob = np.vstack(X_list)
    y_blob = np.concatenate(y_list)
    source = 'NumPy fallback'

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# 1. 散点图：类别结构
sc = axes[0].scatter(X_blob[:, 0], X_blob[:, 1], c=y_blob, cmap='viridis', alpha=0.75)
axes[0].set_title(f'分类数据散点图\n来源: {source}')
axes[0].set_xlabel('特征 1')
axes[0].set_ylabel('特征 2')
plt.colorbar(sc, ax=axes[0], label='类别')

# 2. 每个特征的分布
axes[1].hist(X_blob[:, 0], bins=25, alpha=0.65, label='特征 1')
axes[1].hist(X_blob[:, 1], bins=25, alpha=0.65, label='特征 2')
axes[1].set_title('两个特征的边缘分布')
axes[1].legend()

# 3. 相关矩阵热力图
corr = np.corrcoef(X_blob.T)
im = axes[2].imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
axes[2].set_xticks([0, 1], labels=['特征 1', '特征 2'])
axes[2].set_yticks([0, 1], labels=['特征 1', '特征 2'])
axes[2].set_title('特征相关系数矩阵')
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, f'{corr[i, j]:.2f}', ha='center', va='center', color='black')
plt.colorbar(im, ax=axes[2])

for ax in axes[:2]:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


### 4.1 异常点：为什么一张散点图能救你很多时间？

异常点可能来自：

- 数据录入错误，例如年龄写成 999
- 单位不一致，例如厘米和米混用
- 真实但罕见的样本，例如极端价格、极端传感器读数

异常点不一定都要删除，但必须先发现它们。


In [ ]:
# 给线性数据人为加入异常点，观察回归线受到的影响
x_normal = rng.uniform(-3, 3, 80)
y_normal = 1.8 * x_normal + 0.5 + rng.normal(0, 0.6, size=80)

x_outliers = np.array([2.5, 2.8, -2.6])
y_outliers = np.array([-8.0, -7.0, 8.0])

x_all = np.concatenate([x_normal, x_outliers])
y_all = np.concatenate([y_normal, y_outliers])

coef_normal = np.polyfit(x_normal, y_normal, deg=1)
coef_all = np.polyfit(x_all, y_all, deg=1)
xx = np.linspace(-3.2, 3.2, 200)

plt.figure(figsize=(8, 5))
plt.scatter(x_normal, y_normal, alpha=0.7, label='普通样本')
plt.scatter(x_outliers, y_outliers, color='red', s=80, marker='x', label='异常点')
plt.plot(xx, np.polyval(coef_normal, xx), 'g--', lw=2, label='只用普通样本拟合')
plt.plot(xx, np.polyval(coef_all, xx), 'r-', lw=2, label='包含异常点拟合')
plt.title('异常点会显著影响简单线性模型')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

print(f'无异常点拟合: y = {coef_normal[0]:.2f}x + {coef_normal[1]:.2f}')
print(f'含异常点拟合: y = {coef_all[0]:.2f}x + {coef_all[1]:.2f}')


## 5. 训练过程可视化：看懂 loss 和 accuracy

训练曲线通常比最终指标更有信息量。

| 现象 | 可能含义 |
|------|----------|
| 训练 loss 和验证 loss 都下降 | 模型正在学习 |
| 训练 loss 下降，验证 loss 上升 | 可能过拟合 |
| loss 上下剧烈震荡 | 学习率可能过大，或 batch 太小 |
| loss 几乎不变 | 学习率太小、模型太弱、数据/标签有问题 |
| accuracy 高但 loss 仍高 | 预测类别对了，但置信度可能不稳定 |


In [ ]:
# 模拟训练过程：正常学习 + 后期过拟合
np.random.seed(42)
epochs = 60
epoch = np.arange(1, epochs + 1)

train_loss = 1.8 * np.exp(-0.07 * epoch) + 0.08 + np.random.normal(0, 0.025, epochs)
val_loss = 1.6 * np.exp(-0.055 * epoch) + 0.16 + np.random.normal(0, 0.035, epochs)
val_loss[38:] += np.linspace(0, 0.35, epochs - 38)

train_acc = 0.55 + 0.42 * (1 - np.exp(-0.08 * epoch)) + np.random.normal(0, 0.01, epochs)
val_acc = 0.52 + 0.38 * (1 - np.exp(-0.075 * epoch)) + np.random.normal(0, 0.014, epochs)
val_acc[38:] -= np.linspace(0, 0.08, epochs - 38)

best_epoch = int(np.argmin(val_loss) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

axes[0].plot(epoch, train_loss, label='训练 Loss', lw=2)
axes[0].plot(epoch, val_loss, label='验证 Loss', lw=2)
axes[0].axvline(best_epoch, color='green', linestyle=':', lw=2, label=f'最佳 epoch={best_epoch}')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('训练/验证 Loss：识别过拟合')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epoch, train_acc, label='训练 Accuracy', lw=2)
axes[1].plot(epoch, val_acc, label='验证 Accuracy', lw=2)
axes[1].axvline(best_epoch, color='green', linestyle=':', lw=2, label='早停参考点')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('训练/验证 Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'验证 loss 最低点出现在 epoch {best_epoch}，后续验证 loss 上升是过拟合信号。')


### 5.1 平滑曲线：看趋势，不被噪声误导

真实训练日志常常有噪声，尤其是使用 mini-batch 训练时。可以用移动平均观察趋势。

注意：平滑曲线用于观察，不应替代原始数据；报告中最好同时保留原始曲线的透明版本。


In [ ]:
def moving_average(values, window=5):
    values = np.asarray(values)
    if window <= 1:
        return values
    kernel = np.ones(window) / window
    # mode='same' 保持长度不变，便于和原曲线对齐
    return np.convolve(values, kernel, mode='same')

noisy_loss = train_loss + np.random.normal(0, 0.08, epochs)
smoothed_loss = moving_average(noisy_loss, window=7)

plt.figure(figsize=(9, 4.5))
plt.plot(epoch, noisy_loss, color='C0', alpha=0.35, label='原始 noisy loss')
plt.plot(epoch, smoothed_loss, color='C0', lw=2.5, label='移动平均（window=7）')
plt.plot(epoch, train_loss, color='black', linestyle='--', alpha=0.7, label='潜在真实趋势')
plt.title('平滑曲线帮助观察趋势')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## 6. 模型评估图：不只看一个 accuracy

分类模型常见评估图：

1. **混淆矩阵**：看哪些类别互相混淆
2. **ROC 曲线**：观察不同阈值下 TPR 与 FPR 的权衡
3. **PR 曲线**：类别不均衡时通常比 ROC 更敏感
4. **阈值曲线**：选择业务上合适的 precision / recall 平衡点


In [ ]:
# 混淆矩阵：自己用 NumPy 构造一个三分类例子
confusion = np.array([
    [45,  5,  2],
    [ 3, 40,  7],
    [ 1,  4, 43]
])
class_names = ['类别 A', '类别 B', '类别 C']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

# 原始数量
im0 = axes[0].imshow(confusion, cmap='Blues')
axes[0].set_title('混淆矩阵：样本数量')
axes[0].set_xlabel('预测类别')
axes[0].set_ylabel('真实类别')
axes[0].set_xticks(range(3), class_names)
axes[0].set_yticks(range(3), class_names)
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, str(confusion[i, j]), ha='center', va='center')
plt.colorbar(im0, ax=axes[0])

# 行归一化：每个真实类别中预测比例
confusion_norm = confusion / confusion.sum(axis=1, keepdims=True)
im1 = axes[1].imshow(confusion_norm, cmap='Oranges', vmin=0, vmax=1)
axes[1].set_title('混淆矩阵：按真实类别归一化')
axes[1].set_xlabel('预测类别')
axes[1].set_ylabel('真实类别')
axes[1].set_xticks(range(3), class_names)
axes[1].set_yticks(range(3), class_names)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{confusion_norm[i, j]:.1%}', ha='center', va='center')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

accuracy = np.trace(confusion) / confusion.sum()
print(f'整体准确率: {accuracy:.2%}')
print('但混淆矩阵能进一步告诉我们：类别 B 更容易被误判为类别 C。')


In [ ]:
# ROC 与 PR 曲线：二分类分数的阈值分析（纯 NumPy 实现）
rng = np.random.default_rng(1)
n_pos, n_neg = 180, 420
scores_pos = rng.normal(0.72, 0.16, n_pos)
scores_neg = rng.normal(0.32, 0.18, n_neg)
y_true = np.r_[np.ones(n_pos), np.zeros(n_neg)]
scores = np.clip(np.r_[scores_pos, scores_neg], 0, 1)

thresholds = np.linspace(1, 0, 200)
tpr, fpr, precision, recall = [], [], [], []
for t in thresholds:
    y_pred = scores >= t
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))
    tpr.append(tp / (tp + fn + 1e-12))
    fpr.append(fp / (fp + tn + 1e-12))
    precision.append(tp / (tp + fp + 1e-12))
    recall.append(tp / (tp + fn + 1e-12))

tpr, fpr, precision, recall = map(np.array, [tpr, fpr, precision, recall])
roc_auc = np.trapezoid(tpr[np.argsort(fpr)], np.sort(fpr))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].hist(scores[y_true == 0], bins=25, alpha=0.65, label='负类', density=True)
axes[0].hist(scores[y_true == 1], bins=25, alpha=0.65, label='正类', density=True)
axes[0].set_title('模型分数分布')
axes[0].set_xlabel('预测为正类的分数')
axes[0].legend()

axes[1].plot(fpr, tpr, lw=2, label=f'ROC AUC≈{roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='随机猜测')
axes[1].set_xlabel('FPR：假阳性率')
axes[1].set_ylabel('TPR / Recall：真阳性率')
axes[1].set_title('ROC 曲线')
axes[1].legend()

axes[2].plot(recall, precision, lw=2)
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_ylim(0, 1.05)
axes[2].set_title('PR 曲线')

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 7. 神经网络常见可视化

原脚本中包含两类很重要的 AI 可视化：

- 激活函数：理解非线性、饱和区和梯度传播
- 梯度下降路径：理解优化过程如何在损失地形上移动

下面进一步扩展这些图，并加入导数、对比和直观解释。


In [ ]:
# 激活函数及其导数
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

def relu(x):
    return np.maximum(0, x)

def leaky_relu(x, alpha=0.05):
    return np.where(x > 0, x, alpha * x)

xs = np.linspace(-6, 6, 500)
activations = [
    ('Sigmoid', sigmoid(xs), sigmoid(xs) * (1 - sigmoid(xs))),
    ('Tanh', tanh(xs), 1 - tanh(xs)**2),
    ('ReLU', relu(xs), (xs > 0).astype(float)),
    ('Leaky ReLU', leaky_relu(xs), np.where(xs > 0, 1.0, 0.05)),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharex=True)
for col, (name, y, dy) in enumerate(activations):
    axes[0, col].plot(xs, y, lw=2)
    axes[0, col].axhline(0, color='black', lw=0.5)
    axes[0, col].axvline(0, color='black', lw=0.5)
    axes[0, col].set_title(f'{name} 函数')
    axes[0, col].grid(alpha=0.3)

    axes[1, col].plot(xs, dy, color='C3', lw=2)
    axes[1, col].axhline(0, color='black', lw=0.5)
    axes[1, col].axvline(0, color='black', lw=0.5)
    axes[1, col].set_title(f'{name} 导数')
    axes[1, col].grid(alpha=0.3)

plt.suptitle('激活函数不仅要看函数值，也要看导数：导数决定梯度如何传播', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 7.1 梯度下降路径：在等高线上看优化

等高线图非常适合理解二维损失函数：

- 每条线表示相同的函数值
- 越靠近中心，损失越低
- 梯度下降路径应该逐步走向低处
- 学习率过大时可能震荡，甚至发散


In [ ]:
# 在 f(x,y)=x^2+3y^2 上观察梯度下降路径
def loss_surface(x, y):
    return x**2 + 3*y**2

def grad_surface(point):
    x, y = point
    return np.array([2*x, 6*y])

def gd_path(start=(3.5, 3.5), lr=0.1, steps=25):
    p = np.array(start, dtype=float)
    path = [p.copy()]
    for _ in range(steps):
        p = p - lr * grad_surface(p)
        path.append(p.copy())
    return np.array(path)

x_range = np.linspace(-4, 4, 200)
y_range = np.linspace(-4, 4, 200)
Xg, Yg = np.meshgrid(x_range, y_range)
Zg = loss_surface(Xg, Yg)

learning_rates = [0.03, 0.10, 0.25, 0.38]
fig, axes = plt.subplots(1, 4, figsize=(19, 4.5), sharex=True, sharey=True)

for ax, lr in zip(axes, learning_rates):
    path = gd_path(lr=lr, steps=22)
    ax.contour(Xg, Yg, Zg, levels=25, cmap='coolwarm', alpha=0.9)
    ax.plot(path[:, 0], path[:, 1], 'ko-', markersize=3, linewidth=1.2)
    ax.scatter(path[0, 0], path[0, 1], color='green', s=80, label='起点')
    ax.scatter(path[-1, 0], path[-1, 1], color='red', s=80, marker='*', label='终点')
    ax.set_title(f'学习率 lr={lr}')
    ax.set_aspect('equal')
    ax.grid(alpha=0.25)

axes[0].legend(loc='upper right')
plt.suptitle('梯度下降路径：学习率影响收敛速度和稳定性', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

for lr in learning_rates:
    path = gd_path(lr=lr, steps=22)
    print(f'lr={lr:<4}: 终点=({path[-1,0]: .4f}, {path[-1,1]: .4f}), loss={loss_surface(path[-1,0], path[-1,1]):.6f}')


### 7.2 决策边界：把分类器学到的规则画出来

对二维分类问题，可以把平面上的每个点都喂给模型，得到预测类别，然后画出模型的决策区域。

这里不用复杂模型，而是用“离哪个中心最近就属于哪类”的规则模拟一个分类器。


In [ ]:
# 使用最近中心规则画决策边界
centers = np.array([X_blob[y_blob == k].mean(axis=0) for k in np.unique(y_blob)])

xx, yy = np.meshgrid(
    np.linspace(X_blob[:, 0].min() - 1, X_blob[:, 0].max() + 1, 300),
    np.linspace(X_blob[:, 1].min() - 1, X_blob[:, 1].max() + 1, 300),
)
grid = np.c_[xx.ravel(), yy.ravel()]
# 每个网格点到各中心的距离
dists = ((grid[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
pred = np.argmin(dists, axis=1).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, pred, alpha=0.25, cmap='viridis')
plt.scatter(X_blob[:, 0], X_blob[:, 1], c=y_blob, cmap='viridis', edgecolor='k', linewidth=0.2, alpha=0.75)
plt.scatter(centers[:, 0], centers[:, 1], c='red', marker='*', s=250, label='类别中心')
plt.title('决策边界：模型如何划分特征空间')
plt.xlabel('特征 1')
plt.ylabel('特征 2')
plt.grid(alpha=0.25)
plt.legend()
plt.show()


### 7.3 注意力/权重热力图：imshow 的典型用途

`imshow()` 可以把矩阵画成颜色。AI 中常见用途包括：

- 混淆矩阵
- 图像像素
- CNN 卷积核
- Transformer 注意力权重
- 特征相关矩阵

下面构造一个简化版注意力矩阵：每个 token 更关注自己附近的 token，同时某些词对 `[CLS]` 有额外关注。


In [ ]:
tokens = ['[CLS]', '我', '喜欢', '机器', '学习', '可视化', '[SEP]']
n = len(tokens)
idx = np.arange(n)
attention = np.exp(-0.7 * np.abs(idx[:, None] - idx[None, :]))
attention[:, 0] += np.array([0.2, 0.1, 0.15, 0.35, 0.35, 0.45, 0.1])
attention = attention / attention.sum(axis=1, keepdims=True)

plt.figure(figsize=(7, 6))
im = plt.imshow(attention, cmap='magma')
plt.xticks(range(n), tokens, rotation=45)
plt.yticks(range(n), tokens)
plt.xlabel('被关注的 token')
plt.ylabel('当前 token')
plt.title('示例注意力热力图')
for i in range(n):
    for j in range(n):
        plt.text(j, i, f'{attention[i, j]:.2f}', ha='center', va='center', color='white' if attention[i, j] > 0.2 else 'black', fontsize=8)
plt.colorbar(im, label='attention weight')
plt.tight_layout()
plt.show()


## 8. 绘图工程习惯：让图可复用、可保存、可调试

教程和实验代码中建议养成这些习惯：

1. **封装绘图函数**：相同图表不要复制粘贴多次
2. **固定随机种子**：让图可复现
3. **保存高分辨率图片**：报告用 `dpi=150` 或 `dpi=300`
4. **保存时使用 `bbox_inches='tight'`**：避免标题或标签被裁掉
5. **每个图都有标题、坐标轴和图例**
6. **图表文字服务于结论**：不要只画图，不解释图说明什么


In [ ]:
from pathlib import Path

fig_dir = Path('02_python_ai_tools/generated_figures')
fig_dir.mkdir(parents=True, exist_ok=True)

def plot_training_curves(history, save_path=None):
    """绘制训练曲线的可复用函数。history 是包含 train_loss/val_loss/train_acc/val_acc 的字典。"""
    epochs = np.arange(1, len(history['train_loss']) + 1)
    best_epoch = int(np.argmin(history['val_loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    axes[0].plot(epochs, history['train_loss'], label='train loss')
    axes[0].plot(epochs, history['val_loss'], label='val loss')
    axes[0].axvline(best_epoch, color='green', linestyle=':', label=f'best={best_epoch}')
    axes[0].set_title('Loss 曲线')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, history['train_acc'], label='train acc')
    axes[1].plot(epochs, history['val_acc'], label='val acc')
    axes[1].axvline(best_epoch, color='green', linestyle=':')
    axes[1].set_title('Accuracy 曲线')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'图片已保存到: {save_path}')
    return fig, axes

history = {
    'train_loss': train_loss,
    'val_loss': val_loss,
    'train_acc': train_acc,
    'val_acc': val_acc,
}

fig, axes = plot_training_curves(history, save_path=fig_dir / 'training_curves.png')
plt.show()


### 8.1 常见坑：图画出来了，但误导读者

| 坑 | 后果 | 建议 |
|----|------|------|
| 不标坐标轴 | 读者不知道单位和含义 | 每张图都写 `xlabel/ylabel` |
| y 轴范围随意截断 | 夸大差异 | 截断时明确说明 |
| 多条线颜色太接近 | 难以区分 | 使用颜色 + 线型双重编码 |
| 只画平滑曲线 | 掩盖波动 | 原始曲线透明显示，平滑曲线加粗 |
| 类别不均衡只看 accuracy | 高估模型 | 加混淆矩阵、PR 曲线、召回率 |
| 保存图片后文字被裁剪 | 报告不可用 | `bbox_inches='tight'` |


In [ ]:
# 颜色 + 线型双重编码示例：即使黑白打印也能区分
models = ['Baseline', 'CNN', 'Transformer']
model_curves = {
    'Baseline': 0.78 + 0.06 * (1 - np.exp(-0.06 * epoch)) + rng.normal(0, 0.006, len(epoch)),
    'CNN': 0.72 + 0.16 * (1 - np.exp(-0.08 * epoch)) + rng.normal(0, 0.006, len(epoch)),
    'Transformer': 0.68 + 0.23 * (1 - np.exp(-0.10 * epoch)) + rng.normal(0, 0.006, len(epoch)),
}
styles = {
    'Baseline': dict(color='C0', linestyle='-'),
    'CNN': dict(color='C1', linestyle='--'),
    'Transformer': dict(color='C2', linestyle='-.'),
}

plt.figure(figsize=(9, 4.8))
for name in models:
    plt.plot(epoch, model_curves[name], label=name, linewidth=2, **styles[name])
plt.title('多个模型验证准确率对比')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.ylim(0.75, 0.94)
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## 9. 本节总结与练习

### 关键要点

1. **Matplotlib 的核心对象**：Figure 是画布，Axes 是具体坐标系。
2. **选图比美化更重要**：折线看趋势，散点看关系，直方图看分布，热力图看矩阵。
3. **AI 可视化的主线**：数据探索 → 训练监控 → 模型评估 → 模型调试。
4. **训练曲线能发现问题**：过拟合、学习率过大、学习率过小、训练停滞。
5. **混淆矩阵比 accuracy 更细**：能看出具体哪些类别被误判。
6. **激活函数要看导数**：导数决定梯度是否容易传播。
7. **等高线适合理解优化**：能直观看到梯度下降路径。
8. **工程习惯很重要**：封装函数、保存图片、固定随机种子、标注清楚。

### 绘图检查清单

- [ ] 图是否有标题？
- [ ] x/y 轴是否有标签和单位？
- [ ] 多条曲线是否有图例？
- [ ] 网格、颜色、透明度是否帮助理解？
- [ ] 是否说明了这张图的结论？
- [ ] 如果用于报告，是否保存了高分辨率版本？

### 练习

1. 修改训练曲线模拟代码，让验证 loss 从第 20 个 epoch 就开始过拟合，并观察最佳早停点如何变化。
2. 使用 `np.random.lognormal` 生成偏态分布，画直方图，并尝试对数据取 `log` 后再次绘图。
3. 把混淆矩阵扩展成 5 个类别，并计算每个类别的召回率。
4. 为 ReLU、Leaky ReLU、Sigmoid、Tanh 添加“适合场景/常见问题”的文字说明表格。
5. 在梯度下降示例中修改函数为 `f(x,y)=x^2+10y^2`，观察路径为什么更容易震荡。
6. 将 `plot_training_curves` 函数改造成可以接收任意指标名的通用绘图函数。

下一节建议学习：机器学习基础 —— 从线性回归开始，把数据、损失、梯度和可视化串起来。
